# Steam RPGs: Attention and Positive Review Rates

## A Gamer's Data Exploration

For this project, I decided to analyze video games on Steam. The reason is
simple: **I'm a gamer.**

Outside of my studies, I spend a lot of time on Steam. If you're unfamiliar
with it, Steam is one of the best-known digital distribution platforms for PC
games, where players can buy and launch games.

Over the years, I've played all kinds of games, but I still consider myself an
RPG (role-playing game) enthusiast. At least for now, my two favorite RPGs are
*Cyberpunk 2077* and *Elden Ring*. The former is known for its futuristic world
and demanding graphics requirements; the latter combines action role-playing
with the aesthetics of Western epic fantasy. To me, *Elden Ring* is a classic
that will be hard to surpass for a generation.

I'd recommend both to any RPG fan, although that may be unnecessary—if you're
already an RPG enthusiast, chances are you've played them.

| *Cyberpunk 2077* | *Elden Ring* |
|:---:|:---:|
| [![Cyberpunk 2077 official Steam artwork](https://shared.fastly.steamstatic.com/store_item_assets/steam/apps/1091500/capsule_616x353.jpg)](https://store.steampowered.com/app/1091500/Cyberpunk_2077/) | [![Elden Ring official Steam artwork](https://shared.fastly.steamstatic.com/store_item_assets/steam/apps/1245620/capsule_616x353.jpg)](https://store.steampowered.com/app/1245620/ELDEN_RING/) |

*Images: Promotional artwork from the official Steam store pages.*

As someone from China, I'd also like to recommend a Chinese game:
*Phantom Blade Zero*. Following *Black Myth: Wukong*, it's another Chinese RPG
that has really impressed me. It arrived on Steam in late October 2026. If
you're interested in Chinese martial arts culture and want to wield different
weapons, pull off spectacular moves, and enjoy the thrill of combat, I'd
encourage you to give it a try.

[![Phantom Blade Zero official promotional artwork](https://image.api.playstation.com/vulcan/ap/rnd/202512/1205/4b8cdac83d4ee1c8ee44a27205d1965dac52bcd127d071a0.jpg)](https://store.steampowered.com/app/4115450/Phantom_Blade_Zero/)

*Image: Official PlayStation promotional artwork for Phantom Blade Zero.*

But I digress. Back to Steam. One question has been on my mind:
**On Steam, do RPGs that attract more attention also tend to have higher
positive review rates?**

I use the number of player reviews as a measure of attention. Steam reviews
come from users who have a game in their Steam account, not simply from
anyone browsing its store page. Of course, not everyone who has a game
writes a review. If a game accumulates thousands or tens of thousands of
reviews, that suggests substantial engagement, with many users taking the
time to share their opinions.

Still, having more reviews and having better reviews are clearly not the same
thing. I remember the enormous attention *Cyberpunk 2077* received at launch,
but also the negative Steam reviews from players frustrated by bugs in its
early version. By contrast, I noticed an RPG called *Airborne Empire* this
year. When I checked, it had only about 890 player reviews, yet 85% were
positive, earning Steam's "Very Positive" rating.

Those two examples obviously aren't enough to establish a general pattern.
So I decided to find a larger dataset and take a look.

## Where Does the Data Come From?

I use the [Steam Games Dataset](https://www.kaggle.com/datasets/fronkongames/steam-games-dataset),
compiled by FronkonGames and available on Kaggle. The raw data contains
**125,855 records** of Steam games and related products, including titles,
release dates, genres, prices, and counts of positive and negative reviews.

This isn't a complete official database released by Steam, and it isn't ready
to analyze as-is. The original CSV has formatting issues and includes demos and
other records unsuitable for this project. I cleaned it using Python and
retained **111,762 records**.

Next, I needed to define an RPG. Steam genres aren't mutually exclusive: one
game may be tagged RPG, Action, and Adventure. My rule is straightforward:
**if RPG is one of its genre labels, I count it as an RPG.**

I also needed a study period. During my initial inspection, I noticed gaps in
review coverage for games released in the second half of 2025 and in 2026.
I therefore restrict the main analysis to **RPGs released from 2015 through
2024**. *Airborne Empire* and *Phantom Blade Zero*, which I mentioned above,
aren't part of the study sample.

## Starting With a Decade of RPG Releases

In [1]:
from pathlib import Path

import pandas as pd

# Work from either the site root or the project_1 folder.
project_dir = Path(".")
if not (project_dir / "data/steam_games_clean.csv").is_file():
    project_dir = Path("project_1")

# Read the cleaned dataset and select games tagged RPG.
steam = pd.read_csv(project_dir / "data/steam_games_clean.csv")
steam = steam.rename(columns={"approval_rate": "positive_review_rate"})

is_rpg = (
    steam["genres"]
    .fillna("")
    .str.split(",")
    .apply(lambda labels: "RPG" in [label.strip() for label in labels])
)
in_period = steam["release_year"].between(2015, 2024)
rpg = steam.loc[is_rpg & in_period].copy()

# Count RPGs and the availability of player reviews by release year.
rpg_by_year = rpg.groupby("release_year").agg(
    rpg_games=("appid", "size"),
    with_any_reviews=("review_count", lambda values: (values > 0).sum()),
    with_30_plus_reviews=("review_count", lambda values: (values >= 30).sum()),
)

any_reviews = (rpg["review_count"] > 0).sum()
thirty_plus_reviews = (rpg["review_count"] >= 30).sum()
print(f"RPGs released from 2015 to 2024: {len(rpg):,}")
print(f"With at least one recorded review: {any_reviews:,}")
print(f"With at least 30 recorded reviews: {thirty_plus_reviews:,}")
rpg_by_year

RPGs released from 2015 to 2024: 16,152
With at least one recorded review: 13,263
With at least 30 recorded reviews: 6,931


,rpg_games,with_any_reviews,with_30_plus_reviews
release_year,,,
2015,484,480,432
2016,580,580,465
2017,833,820,526
2018,1167,1131,653
2019,1321,1217,638
2020,1532,1385,720
2021,1883,1560,774
2022,2133,1678,767
2023,2633,1938,878


The sample contains **16,152 RPGs**. Of these, **13,263** have at least one
recorded review, and **6,931** have at least 30.

Another thing caught my attention: the number of games differs greatly across
release years. The dataset includes **484 RPGs released in 2015** but **3,586
released in 2024**. Of the 2015 games, 480 have at least one recorded review;
for the 2024 games, that number is 2,474.

There are at least two issues here. First, this is a snapshot. Players continue
to post Steam reviews, but my downloaded CSV does not automatically update.
Second, a game released in 2015 has had much longer to accumulate reviews than
one released in 2024.

## A Few Hits and a Lot of Ordinary RPGs

Before comparing attention with positive review rates, I want to see how review counts
are distributed.

In [2]:
# Summarize the number of reviews using pandas.
reviews = rpg["review_count"]

print(f"Mean: {reviews.mean():,.2f}")
print(f"Median: {reviews.median():,.2f}")
print(f"Mode: {reviews.mode().iloc[0]:,}")
print(f"Maximum: {reviews.max():,}")
print(f"Games with zero recorded reviews: {(reviews == 0).sum():,}")

Mean: 1,868.38
Median: 18.00
Mode: 0
Maximum: 1,227,784
Games with zero recorded reviews: 2,889


The results are rather striking.

An RPG has about **1,868 reviews on average**, but the median is only **18**,
and the mode is **0**. There are **2,889 games** with no recorded reviews.
In other words, the mean is more than a hundred times the median!

It's not hard to see why. A small number of very popular games attract huge
numbers of reviews and pull the mean upward. For most RPGs, 1,868 reviews is
nowhere near what they actually have.

To check these numbers, I independently read the same CSV and calculated the
mean, median, and mode again using **only Python's standard library**—without
pandas or the `statistics` module.

In [3]:
import csv
from pathlib import Path

# Independently find the CSV without using any pandas variable.
pure_data_path = Path("data/steam_games_clean.csv")
if not pure_data_path.is_file():
    pure_data_path = Path("project_1") / pure_data_path

review_counts = []
with pure_data_path.open("r", encoding="utf-8", newline="") as source_file:
    reader = csv.DictReader(source_file)
    for row in reader:
        genres = [genre.strip() for genre in row["genres"].split(",")]
        year = int(row["release_year"])
        if "RPG" in genres and 2015 <= year <= 2024:
            review_counts.append(int(row["review_count"]))

# Mean: sum of the observations divided by their count.
mean_pure = sum(review_counts) / len(review_counts)

# Median: sort the values and inspect the middle position(s).
sorted_counts = sorted(review_counts)
count_games = len(sorted_counts)
if count_games % 2:
    median_pure = sorted_counts[count_games // 2]
else:
    middle = count_games // 2
    median_pure = (sorted_counts[middle - 1] + sorted_counts[middle]) / 2

# Mode: use a dictionary to count how often each value appears.
frequencies = {}
for value in review_counts:
    frequencies[value] = frequencies.get(value, 0) + 1
mode_pure = max(frequencies, key=frequencies.get)

print(f"Pure Python mean: {mean_pure:,.2f}")
print(f"Pure Python median: {median_pure:,.2f}")
print(f"Pure Python mode: {mode_pure:,}")

Pure Python mean: 1,868.38
Pure Python median: 18.00
Pure Python mode: 0


The results match those from pandas exactly.

But those three summary statistics still don't show where most games fall
along the distribution. So I divided the review counts into groups and drew
a text-based bar chart using **Python's standard library**, without an
external plotting library.

In [4]:
# Build a character bar chart with Python's standard library only.
review_groups = {
    "0": 0,
    "1-9": 0,
    "10-29": 0,
    "30-99": 0,
    "100-499": 0,
    "500-1,999": 0,
    "2,000+": 0,
}

for review_count in review_counts:
    if review_count == 0:
        review_groups["0"] += 1
    elif review_count < 10:
        review_groups["1-9"] += 1
    elif review_count < 30:
        review_groups["10-29"] += 1
    elif review_count < 100:
        review_groups["30-99"] += 1
    elif review_count < 500:
        review_groups["100-499"] += 1
    elif review_count < 2000:
        review_groups["500-1,999"] += 1
    else:
        review_groups["2,000+"] += 1

# The bar width is capped to fit a narrow laptop display.
max_group_size = max(review_groups.values())
bar_width = 30
print("Steam RPG Review Counts by Group (2015-2024)")
print("Bar length = number of games; groups = number of reviews.")
print(f"Total RPG games: {len(review_counts):,}\n")

for label, group_count in review_groups.items():
    bar = "█" * round(group_count / max_group_size * bar_width)
    percentage = 100 * group_count / len(review_counts)
    print(f"{label:>9} | {bar:<30} {group_count:>5,} ({percentage:4.1f}%)")

Steam RPG Review Counts by Group (2015-2024)
Bar length = number of games; groups = number of reviews.
Total RPG games: 16,152

        0 | ████████████████████████       2,889 (17.9%)
      1-9 | ██████████████████████████████ 3,671 (22.7%)
    10-29 | ██████████████████████         2,661 (16.5%)
    30-99 | ████████████████████           2,452 (15.2%)
  100-499 | ███████████████████            2,297 (14.2%)
500-1,999 | █████████                      1,148 ( 7.1%)
   2,000+ | ████████                       1,034 ( 6.4%)


The chart makes the pattern clear: **3,671 RPGs have only 1–9 reviews**,
while just **1,034 have at least 2,000**.

Another number is worth noticing: **9,221 games, or 57.1% of the sample,
have fewer than 30 reviews**.

That matters for what comes next. A game with only two reviews can have a
positive review rate of 0%, 50%, or 100%. We can calculate those percentages, of
course, but they are hardly stable measures of player opinion.

So, from this point onward, I'll analyze games with **at least 30 reviews**.
This is simply a threshold I chose to reduce the instability of extremely
small review samples. The cost is obvious: it excludes more than half of the
RPGs. I'm no longer discussing all RPGs, but games that have accumulated at
least some player reviews.

## Review Counts and Positive Review Rates

Now, back to attention and positive review rates.

For each game, I calculate the positive review rate as positive reviews divided
by the sum of positive and negative reviews. I then divide the **6,931 RPGs**
into groups based on review counts and compare the **median positive review rate**
across groups.

I use the median of individual games' positive review rates rather than combining
all the reviews within each group. This way, **each game** is an observation,
and the statistic describes a typical game in the group.

In [5]:
# Compare median positive review rates by review-count group.
rpg_analysis = rpg.loc[rpg["review_count"] >= 30].copy()

review_bins = [29, 99, 499, 1999, 9999, float("inf")]
review_labels = ["30-99", "100-499", "500-1,999", "2,000-9,999", "10,000+"]
rpg_analysis["review_group"] = pd.cut(
    rpg_analysis["review_count"],
    bins=review_bins,
    labels=review_labels,
)

group_summary = rpg_analysis.groupby("review_group", observed=True).agg(
    games=("appid", "size"),
    median_positive_review_rate=("positive_review_rate", "median"),
)

# Convert proportions to percentages for easier reading.
group_summary["median_positive_review_rate_pct"] = (
    100 * group_summary["median_positive_review_rate"]
).round(1)
print(f"RPGs included: {len(rpg_analysis):,}")
print("Median positive review rate by review-count group (%):")
group_summary[["games", "median_positive_review_rate_pct"]].rename(
    columns={
        "games": "RPGs",
        "median_positive_review_rate_pct": ("Median positive review rate (%)"),
    }
)

RPGs included: 6,931
Median positive review rate by review-count group (%):


,RPGs,Median positive review rate (%)
review_group,,
30-99,2452,77.4
100-499,2297,79.0
"500-1,999",1148,81.1
"2,000-9,999",682,84.9
"10,000+",352,86.9


The result matches my intuition. Among RPGs with **30–99 reviews**, the median
positive review rate is approximately **77.4%**. It rises to **81.1%** for
games with **500–1,999 reviews**, and to **86.9%** for those with **10,000
reviews or more**.

So, in this grouped comparison, RPGs with more reviews also tend to have
higher positive review rates.

That doesn't mean attention *causes* higher positive review rates. Perhaps well-reviewed
games attract more players, or perhaps other factors, such as game quality,
affect both measures. These data cannot distinguish among those explanations.

What interests me more is that this comparison mixes ten release years.
Games from different years have had different amounts of time to accumulate
reviews. I wanted to see what would happen if I separated them.

## Separating Games by Release Year

I calculated **Spearman's rank correlation coefficient (ρ)** between review counts and
positive review rates separately for each release year.

This statistic compares the rankings of two variables. Here, it tells me
whether RPGs with higher review counts within a given release year also tend
to rank higher in positive review rates. A positive coefficient means the rankings tend
to move together; a value near zero indicates little monotonic association.

In [6]:
# Calculate Spearman correlations within each release-year group.
yearly_rows = []

for release_year, games in rpg_analysis.groupby("release_year", sort=True):
    # Spearman's rho is the Pearson correlation between ranks.
    review_ranks = games["review_count"].rank()
    positive_review_ranks = games["positive_review_rate"].rank()
    rho = review_ranks.corr(positive_review_ranks)
    yearly_rows.append(
        {
            "Release year": int(release_year),
            "RPGs": len(games),
            "Spearman ρ": rho,
        }
    )

yearly_summary = pd.DataFrame(yearly_rows)
yearly_summary.round({"Spearman ρ": 3})

,Release year,RPGs,Spearman ρ
0,2015,432,0.399
1,2016,465,0.338
2,2017,526,0.247
3,2018,653,0.306
4,2019,638,0.160
5,2020,720,0.245
6,2021,774,0.229
7,2022,767,0.100
8,2023,878,0.048
9,2024,1078,-0.023


This time, the results are less tidy. For RPGs released in **2015**, the
correlation is approximately **0.399**; for **2016**, it's **0.338**. In those
groups, games with more reviews tend to have higher positive review rates.

But things look different for more recent release years. The coefficient is
**0.100 in 2022**, **0.048 in 2023**, and **−0.023 in 2024**—almost zero.

The coefficients don't fall every single year: there are rebounds in 2018 and
2020, for example. Still, the earlier release-year groups tend to show a
stronger positive association than the more recent ones.

This is the finding I find most interesting. Looking only at the grouped
comparison across all years, it would be easy to conclude that RPGs with more
reviews generally have higher positive review rates. Separating games by release
year shows that the pattern is not nearly as consistent.

I should emphasize that these are **comparisons between different groups of
games released in different years**, not a record of the same games changing
over time. So these differences cannot establish that the relationship between
attention and positive review rates has been weakening over time.

Why do the release-year groups differ? I don't have an answer yet. The time
available to accumulate reviews, the gaming market, differences in the mix of
games, and which players choose to write reviews may all matter.

Anyway, that's my initial analysis. I hope it offers at least a starting
point for someone who wants to dig further.

---

*Source and acknowledgment: This analysis uses the Steam Games Dataset
compiled by FronkonGames. ChatGPT was used to assist in the writing and checking of the Python code.